<a href="https://colab.research.google.com/github/BytePhilosopher/Football_data_tracking/blob/main/Football_Tracking_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football Tracking Pipeline

**Before running:** Make sure the runtime is set to **GPU** (Runtime → Change runtime type → T4 GPU).

### What this notebook does
1. Clones the repo and installs dependencies
2. Mounts Google Drive to load your video (`168.mp4`) and model (`best.pt`)
3. Runs the full tracking pipeline
4. Copies outputs back to Drive
5. (Optional) Pushes results to GitHub

## Step 1 — Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU found. Go to Runtime → Change runtime type → T4 GPU')

## Step 2 — Clone repo & install dependencies

In [ ]:
import os

REPO_URL  = 'https://github.com/BytePhilosopher/Football_data_tracking.git'
REPO_DIR  = '/content/football_tracking_project'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest changes if repo already exists
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q ultralytics supervision scikit-learn tqdm opencv-python-headless

## Step 3 — Mount Google Drive

Your Drive should have:
```
MyDrive/
  football_tracking/
    168.mp4        ← input video
    best.pt        ← YOLO model weights
```
Adjust `DRIVE_FOLDER` below if your folder is named differently.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

# ── Configure these to match your Drive layout ──────────────────────────────
DRIVE_FOLDER = '/content/drive/MyDrive/football_tracking'
VIDEO_NAME   = '168.mp4'
MODEL_NAME   = 'best.pt'
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs('data/raw',        exist_ok=True)
os.makedirs('data/processed',  exist_ok=True)
os.makedirs('data/annotations',exist_ok=True)
os.makedirs('models',          exist_ok=True)

# Copy video
src_video = os.path.join(DRIVE_FOLDER, VIDEO_NAME)
dst_video = os.path.join('data/raw', VIDEO_NAME)
if not os.path.exists(dst_video):
    print(f'Copying video from Drive...  ({src_video})')
    shutil.copy(src_video, dst_video)
    print('Done.')
else:
    print('Video already present, skipping copy.')

# Copy model
src_model = os.path.join(DRIVE_FOLDER, MODEL_NAME)
dst_model = os.path.join('models', MODEL_NAME)
if not os.path.exists(dst_model):
    print(f'Copying model from Drive...  ({src_model})')
    shutil.copy(src_model, dst_model)
    print('Done.')
else:
    print('Model already present, skipping copy.')

print('\nFiles ready:')
print(f'  Video : {dst_video}  ({os.path.getsize(dst_video)/1e6:.1f} MB)')
print(f'  Model : {dst_model}  ({os.path.getsize(dst_model)/1e6:.1f} MB)')

## Step 4 — Verify pipeline config

Make sure the paths in `main.py` match the files we just copied.

In [ ]:
# Show the config section of main.py so you can verify paths
!head -25 main.py

## Step 5 — Run the pipeline

In [ ]:
# Make sure we're in the right directory before running
os.chdir(REPO_DIR)
print('Running from:', os.getcwd())
!python main.py

## Step 6 — Preview output video (inline)

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import glob

# Find the output video
candidates = glob.glob('data/processed/*_tracked.mp4')
if not candidates:
    candidates = glob.glob('data/processed/*.mp4')

if candidates:
    output_video = candidates[0]
    print('Previewing:', output_video)
    with open(output_video, 'rb') as f:
        video_data = b64encode(f.read()).decode()
    HTML(f'''
    <video width="900" controls>
      <source src="data:video/mp4;base64,{video_data}" type="video/mp4">
    </video>
    ''')
else:
    print('No output video found in data/processed/')

## Step 7 — Copy outputs back to Google Drive

In [ ]:
import shutil, glob, os

DRIVE_OUTPUT = os.path.join(DRIVE_FOLDER, 'outputs')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Copy output video
for f in glob.glob('data/processed/*.mp4'):
    dest = os.path.join(DRIVE_OUTPUT, os.path.basename(f))
    shutil.copy(f, dest)
    print(f'Saved video → {dest}')

# Copy CSVs
for f in glob.glob('data/annotations/*.csv'):
    dest = os.path.join(DRIVE_OUTPUT, os.path.basename(f))
    shutil.copy(f, dest)
    print(f'Saved CSV   → {dest}')

print('\nAll outputs saved to Google Drive.')

In [ ]:
import zipfile, glob
from google.colab import files

zip_path = 'pipeline_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for csv_file in glob.glob('data/annotations/*.csv'):
        zf.write(csv_file, os.path.basename(csv_file))
        print(f'  Added: {os.path.basename(csv_file)}')

print(f'\nCreated {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)')
print('Downloading...')
files.download(zip_path)

## Step 7b — Download results for Streamlit app

Run this cell to download a zip of all CSV outputs. Upload these files on the **Analysis** page of the deployed Streamlit app to visualize results without needing a GPU.

## Step 8 (Optional) — Push results to GitHub

**Setup (one time):**
1. Go to GitHub → Settings → Developer settings → Personal access tokens → Tokens (classic)
2. Generate a token with `repo` scope
3. In Colab, click the **🔑 Secrets** icon in the left sidebar
4. Add a secret named `GH_PAT` with your token value
5. Enable **Notebook access** for that secret

In [ ]:
from google.colab import userdata

try:
    GH_PAT = userdata.get('GH_PAT')
    GH_USER  = 'BytePhilosopher'
    GH_EMAIL = 'aberayostina@gmail.com'
    GH_REPO  = 'Football_data_tracking'

    !git config --global user.email "{GH_EMAIL}"
    !git config --global user.name  "{GH_USER}"

    # Stage only CSVs and the notebook — not large video files
    !git add data/annotations/*.csv Football_Tracking_Colab.ipynb
    !git diff --cached --quiet || git commit -m "Colab run: updated tracking results"

    push_url = f'https://{GH_PAT}@github.com/{GH_USER}/{GH_REPO}.git'
    !git push {push_url} main
    print('Pushed to GitHub successfully.')

except Exception as e:
    print(f'GitHub push skipped: {e}')
    print('To enable push, add your GitHub PAT as a Colab secret named GH_PAT.')